In [ ]:
# Example 1: import primary public submodules and inspect their attributes.
from opentelemetry.util.genai import types, version

print(f"types module: {types}")
print(f"version module: {version}")

type_names = [name for name in dir(types) if not name.startswith('_')]
print(f"types exports ({len(type_names)}): {type_names[:15]}")

assert len(type_names) > 0, "types module should expose public names"


In [ ]:
# Example 2: instantiate core GenAI types (LLMInvocation / message parts).
from opentelemetry.util.genai import types

# Find a usable message / part class
candidates = ['Text', 'InputMessage', 'OutputMessage', 'LLMInvocation']
found = {name: getattr(types, name, None) for name in candidates}
found = {k: v for k, v in found.items() if v is not None}

for name, cls in found.items():
    print(f"{name}: {cls}")

assert len(found) > 0, f"Expected at least one of {candidates} in types"


In [ ]:
# Example 3: environment-variable configuration helpers.
import os
from opentelemetry.util.genai import environment_variables as env_vars

var_names = [n for n in dir(env_vars) if n.startswith('OTEL_')]
print(f"Discovered {len(var_names)} OTEL_* env var constants")
for name in var_names[:10]:
    val = getattr(env_vars, name)
    print(f"  {name} = {val!r}")

assert len(var_names) > 0, "environment_variables should declare OTEL_* constants"


In [ ]:
# Example 4: the handler records an offline inference span
from opentelemetry.util import genai

public_names = [n for n in dir(genai) if not n.startswith('_')]
print(f"opentelemetry.util.genai public names: {public_names}")

import importlib
imported = []
submodules = ('handler', 'utils', 'types', 'environment_variables')
for sub in submodules:
    mod = importlib.import_module(f'opentelemetry.util.genai.{sub}')
    imported.append((sub, mod))

print(f"\nImported {len(imported)} submodules:")
for name, mod in imported:
    print(f"  {name}: {mod.__name__}")

assert [mod.__name__ for _, mod in imported] == [f'opentelemetry.util.genai.{s}' for s in submodules]

from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, in_memory_span_exporter
from opentelemetry.util.genai.handler import TelemetryHandler

exporter = in_memory_span_exporter.InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(exporter))
with TelemetryHandler(tracer_provider=provider).inference("openai", request_model="gpt-4") as invocation:
    invocation.response_model_name = "gpt-4-0613"
(span,) = exporter.get_finished_spans()
print(f"span {span.name!r} attributes {dict(span.attributes)}")
assert span.name == "chat gpt-4", span.name
assert {"gen_ai.operation.name": "chat", "gen_ai.provider.name": "openai", "gen_ai.request.model": "gpt-4",
        "gen_ai.response.model": "gpt-4-0613"}.items() <= dict(span.attributes).items(), dict(span.attributes)

In [ ]:
# Example 5: verify the package interoperates with core OpenTelemetry API.
from opentelemetry import trace
from opentelemetry.util.genai import types

tracer = trace.get_tracer("genai-smoke-test")
with tracer.start_as_current_span("gen_ai.request") as span:
    span.set_attribute("gen_ai.system", "openai")
    span.set_attribute("gen_ai.request.model", "gpt-4")
    ctx = span.get_span_context()
    print(f"span trace_id={ctx.trace_id:032x} span_id={ctx.span_id:016x}")

# Sanity: types module is loadable alongside trace API
assert types is not None
assert hasattr(trace, 'get_tracer')
print("\nopentelemetry.util.genai integrates with opentelemetry.trace OK")
